# Watch a training run in a notebook

This notebook trains a small transformer on synthetic data and shows four live views of it while it trains: the loss as a series, the attention weights as a heatmap with a choice of layer, sequence and head, the model's 3-D embedding of an evaluation set as points coloured by class, and the model's predictions for the same evaluation set as a heatmap. A selection in the embedding marks the same sequences in the predictions. A panel of controls pauses and steps the training loop, changes its learning rate and saves a checkpoint while it trains. The same views and controls open in a browser tab.

It needs the `caliper` package with the extra `caliper[notebook]`, PyTorch, and a notebook front end such as JupyterLab or VS Code. On an Apple Silicon Mac the model trains on the GPU through MPS. The tutorial [Watch a training run in a notebook](../../docs/tutorials/watch-a-training-run-in-a-notebook.md) walks through each cell, and the guide [Control a running script](../../docs/guides/control-a-running-script.md) explains the controls.

`caliper.serve()` starts a server for this notebook's views on 127.0.0.1 and prints a link that carries a session token. Open the link in a browser tab to see every view of the notebook in a grid. The server accepts connections only from this machine and only with the token.

In [ ]:
import math
import os
import tempfile
import time

import torch
import torch.nn as nn
import torch.nn.functional as F

import caliper

server = caliper.serve()

## The data

Each example is a sequence of 24 random tokens. One position holds a marker token, and the token right after the marker decides the class, one of six. The six class tokens also appear at random elsewhere in the sequence, so the model has to find the marker to get the class right. The data is generated on the device at every step, so there is nothing to download.

The first eight sequences of every batch are the same fixed probe sequences. The attention heatmap shows these, so it follows the same inputs from step to step.

`STEPS` sets the length of the run. Set the environment variable `CALIPER_NOTEBOOK_STEPS` to shorten it, for example to 200 for a quick test.

In [ ]:
STEPS = int(os.environ.get("CALIPER_NOTEBOOK_STEPS", "8000"))
BATCH = 256
VOCAB, LENGTH, CLASSES, PROBES = 32, 24, 6, 8
START, MARKER, FIRST_CLASS_TOKEN = 0, 1, 2     # the class tokens are 2 to 7

if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
    print("MPS is not available, so the model trains on the CPU, more slowly.")

torch.manual_seed(0)


def make_batch(n):
    x = torch.randint(2, VOCAB, (n, LENGTH), device=device)
    y = torch.randint(0, CLASSES, (n,), device=device)
    where = torch.randint(1, LENGTH - 1, (n,), device=device)
    rows = torch.arange(n, device=device)
    x[rows, where] = MARKER
    x[rows, where + 1] = FIRST_CLASS_TOKEN + y
    x[:, 0] = START
    return x, y


probe_x, probe_y = make_batch(PROBES)

## The model

The model is a transformer with two layers of four attention heads each. It computes attention explicitly, so each layer returns its attention weights. The output at the start token passes through a 3-D embedding layer before the classifier, so the points the notebook shows are the model's own embedding and need no projection.

In [ ]:
class Block(nn.Module):
    def __init__(self, width, heads):
        super().__init__()
        self.heads = heads
        self.norm1, self.norm2 = nn.LayerNorm(width), nn.LayerNorm(width)
        self.qkv, self.out = nn.Linear(width, 3 * width), nn.Linear(width, width)
        self.mlp = nn.Sequential(nn.Linear(width, 4 * width), nn.GELU(), nn.Linear(4 * width, width))

    def forward(self, h):
        b, t, w = h.shape
        q, k, v = self.qkv(self.norm1(h)).view(b, t, 3, self.heads, w // self.heads).permute(2, 0, 3, 1, 4)
        weights = (q @ k.transpose(-1, -2) / math.sqrt(w // self.heads)).softmax(-1)  # (batch, head, query, key)
        h = h + self.out((weights @ v).transpose(1, 2).reshape(b, t, w))
        return h + self.mlp(self.norm2(h)), weights


class TinyTransformer(nn.Module):
    def __init__(self, width=64, heads=4, layers=2):
        super().__init__()
        self.tokens = nn.Embedding(VOCAB, width)
        self.positions = nn.Parameter(0.02 * torch.randn(LENGTH, width))
        self.blocks = nn.ModuleList(Block(width, heads) for _ in range(layers))
        self.norm = nn.LayerNorm(width)
        self.embed = nn.Linear(width, 3)
        self.classify = nn.Linear(3, CLASSES)

    def forward(self, x):
        h = self.tokens(x) + self.positions
        attention = []
        for block in self.blocks:
            h, weights = block(h)
            attention.append(weights)
        z = self.embed(self.norm(h[:, 0]))
        return self.classify(z), z, torch.stack(attention)  # attention: (layer, batch, head, query, key)


model = TinyTransformer().to(device)
opt = torch.optim.AdamW(model.parameters(), lr=2e-3)

## Controls

`caliper.param` registers the learning rate as a value that the widgets and the browser tab can change while the model trains, between 0.0001 and 0.1 on a log scale. `caliper.action` registers a function that a viewer runs with a button; this one saves the model's weights. Both act at `caliper.step()`, the checkpoint that the training loop calls once per step, so a step never sees a change halfway through.

The evaluation set is 512 fixed sequences. The embedding and the predictions both show these sequences, and both name them as the item space `eval`, so selecting points in the embedding marks the same sequences in the predictions heatmap. `caliper.items` adds the class and the marker position of each sequence, which a viewer shows in a small table for the selected sequences.

In [ ]:
EVAL = 512
eval_x, eval_y = make_batch(EVAL)
eval_marker = (eval_x == MARKER).int().argmax(1)
evaluation = caliper.items("eval", details={"class": eval_y, "marker at": eval_marker})

lr = caliper.param("lr", 2e-3, min=1e-4, max=1e-1, log=True, help="learning rate")
CHECKPOINT = os.path.join(tempfile.gettempdir(), "caliper-live-training.pt")


@caliper.action("save checkpoint", help="saves the model's weights")
def save_checkpoint():
    torch.save(model.state_dict(), CHECKPOINT)
    return f"saved {CHECKPOINT}"

## The widgets

`caliper.widget(name, kind)` shows one view in the notebook. Each widget reads the server's stream directly rather than the notebook's own message channel, so it keeps updating while the training cell below runs. Every widget pans, zooms and shows the value under the pointer, and the attention heatmap has a picker for the layer, the sequence and the head. The kind is optional; without it, the widget takes the kind from the stream.

`caliper.controls()` is the panel of controls: the run state with the buttons Pause, Resume and Step, the learning rate, and the action. The header of each view widget shows the run state and the same buttons.

In [ ]:
from ipywidgets import HBox, VBox

loss_widget = caliper.widget("loss", "series", height=200)
attention_widget = caliper.widget("attention", "heatmap")
embedding_widget = caliper.widget("embedding", "points")
predictions_widget = caliper.widget("predictions", "heatmap", height=160)
attention_widget.layout.flex = embedding_widget.layout.flex = "1 1 0"

VBox([caliper.controls(), loss_widget, HBox([attention_widget, embedding_widget]), predictions_widget])

## Training

Each step starts with `caliper.step()`, where a pause from a viewer holds the loop, a new learning rate arrives and an action runs, and then copies the learning rate into the optimizer. It calls `caliper.series` for the loss and `caliper.show` for the attention weights of the probe sequences. Every tenth step it runs the evaluation set through the model and calls `caliper.points` for its embedding and `caliper.show` for its predicted class probabilities, both with `items` naming the item space `eval`. The loop passes `loss.detach()` rather than `loss.item()`, so it never waits for the GPU. The calls return at once, and the server sends the newest snapshot of each view to the widgets and tabs that show it.

`dims` names the dimensions of the attention tensor, so that a viewer can choose a data view of it, such as one layer, one sequence and one head. `data_view` sets the default data view: layer 1, sequence 0 and head 0. The data view is computed on the device, and only the resulting 24 by 24 heatmap is sent.

The loss usually stays on a plateau for a while and then falls quickly once the model has learned to find the marker. At that point the embedding separates into six clusters, one for each class, the predictions sharpen to one class per sequence, and in layer 1 the row of the start token, row 0, puts its weight on the class token after the marker.

In [ ]:
ATTENTION_DIMS = ("layer", "sequence", "head", "query", "key")
ATTENTION_DEFAULT = {"index": {"layer": 1, "sequence": 0, "head": 0}}


def train(steps):
    start = time.perf_counter()
    for step in range(steps):
        caliper.step()  # pauses, the new learning rate and actions take effect here
        for group in opt.param_groups:
            group["lr"] = lr.value
        x, y = make_batch(BATCH)
        x[:PROBES] = probe_x  # the same probe sequences at the top of every batch
        y[:PROBES] = probe_y
        logits, z, attention = model(x)
        loss = F.cross_entropy(logits, y)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()

        caliper.series(loss.detach(), "loss")
        caliper.show(attention[:, :PROBES], "attention", cmap="magma",
                     dims=ATTENTION_DIMS, data_view=ATTENTION_DEFAULT)
        if step % 10 == 0:
            with torch.no_grad():
                eval_logits, eval_z, _ = model(eval_x)
            caliper.points(eval_z, eval_y, "embedding", classes=CLASSES, items="eval")
            caliper.show(eval_logits.softmax(1).T, "predictions", dims=("class", "sequence"),
                         items={"sequence": "eval"})

    final_loss = loss.item()  # one wait for the GPU, after the loop
    accuracy = (logits.argmax(1) == y).float().mean().item()
    seconds = time.perf_counter() - start
    print(f"{steps} steps in {seconds:.1f} s ({steps / seconds:.0f} steps/s), "
          f"loss {final_loss:.3f}, accuracy on the last batch {accuracy:.0%}")


train(STEPS)

## Selections and data views from Python

Select a few points in the embedding widget, then run the next cell. The widget's `selection` holds the latest selection made in the browser, and `evaluation.selection` holds the sequences of the evaluation set that are selected now in any view. Setting `data_view` from Python asks for another data view, here the largest weight over the four heads of layer 0, where the token after the marker learns to attend to the marker. The kernel receives a widget's changes only while no cell runs, so the cell reads the selection first, and then trains for a few more seconds so that the heatmap shows the new data view.

In [ ]:
print("selection:", embedding_widget.selection)
print("selected sequences:", evaluation.selection[:20])

attention_widget.data_view = {"index": {"layer": 0, "sequence": 0}, "reduce": {"head": "max"}}
train(max(1, STEPS // 10))

## Closing

`server.close()` stops the server and ends the stream to every widget and browser tab.

In [ ]:
server.close()